# Product Knowledge Base Builder
**Flow:**
1. Mount Google Drive → download Product List Excel + txt review files
2. Scrape product descriptions and Reddit discourse for each product
3. Parse pre-collected `.txt` review files (Amazon / Target / Walmart) and inject into knowledge base
4. Save one `knowledge_base.json`

## 1. Install Dependencies *(run once)*

In [11]:
# import subprocess, sys

# pkgs = ["requests", "httpx", "beautifulsoup4", "playwright",
#         "fake-useragent", "lxml", "pandas", "openpyxl", "tqdm", "nest_asyncio", "gdown"]
# subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "-q"])
# subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])

## 2. Imports & Configuration

In [4]:
import json, re, time, random, logging, asyncio
from pathlib import Path
from urllib.parse import urlparse
from typing import Optional

import gdown
import httpx
import pandas as pd
import requests
import nest_asyncio
from bs4 import BeautifulSoup
from fake_useragent import UserAgent
from playwright.async_api import async_playwright
from tqdm import tqdm

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

# ── Config ───────────────────────────────────────────────────────────────
LOCAL_DIR   = Path("/tmp/kb_data")   # where Drive files are downloaded
LOCAL_DIR.mkdir(exist_ok=True)

OUTPUT_JSON = "knowledge_base.json"

MAX_REVIEWS_PER_SOURCE = 50
MIN_DELAY = 2.0
MAX_DELAY = 5.0

PLAYWRIGHT_DOMAINS = {"amazon.com", "target.com", "bestbuy.com", "walmart.com"}

ua = UserAgent()

## 3. Google Drive — File IDs
For each file, open it in Drive → click the 3-dot menu → **Share** → **Copy link**.
The ID is the long string between `/d/` and `/view` in the URL.
All files must be set to **'Anyone with the link can view'**.

In [12]:
# ── Paste your file IDs here ─────────────────────────────────────────────

# Product List Excel (the Google Sheet exported as .xlsx, or a real .xlsx in Drive)
EXCEL_FILE_ID = "YOUR_PRODUCT_LIST_EXCEL_FILE_ID"

# txt files folder structure mirrors Drive:
#   txt files/amazon/<product>_amazon.txt
#   txt files/target/<product>_target.txt
#   txt files/walmart/<product>_walmart.txt
#
# Add an entry for every txt file in your Drive folders.
# Format: "local_filename": "drive_file_id"

TXT_FILE_IDS = {
    # amazon
    "airpods4_amazon.txt":          "FILE_ID",
    "candle_amazon.txt":             "FILE_ID",
    "coffemaker_amazon.txt":         "FILE_ID",
    "crocs_amazon.txt":              "FILE_ID",
    "electrictoothbrush_amazon.txt": "FILE_ID",
    "owala_amazon.txt":              "FILE_ID",
    "oxiclean_amazon.txt":           "FILE_ID",
    "pimplepatches_amazon.txt":      "FILE_ID",
    "proteinbar_amazon.txt":         "FILE_ID",
    "vacuum_amazon.txt":             "FILE_ID",
    # target
    "airpods4_target.txt":          "FILE_ID",
    "candle_target.txt":             "FILE_ID",
    "coffemaker_target.txt":         "FILE_ID",
    "crocs_target.txt":              "FILE_ID",
    "electrictoothbrush_target.txt": "FILE_ID",
    "owala_target.txt":              "FILE_ID",
    "oxiclean_target.txt":           "FILE_ID",
    "pimplepatches_target.txt":      "FILE_ID",
    "proteinbar_target.txt":         "FILE_ID",
    "vacuum_target.txt":             "FILE_ID",
    # walmart
    "airpods4_walmart.txt":          "FILE_ID",
    "candle_walmart.txt":             "FILE_ID",
    "coffemaker_walmart.txt":         "FILE_ID",
    "crocs_walmart.txt":              "FILE_ID",
    "electrictoothbrush_walmart.txt": "FILE_ID",
    "owala_walmart.txt":              "FILE_ID",
    "oxiclean_walmart.txt":           "FILE_ID",
    "pimplepatches_walmart.txt":      "FILE_ID",
    "proteinbar_walmart.txt":         "FILE_ID",
    "vacuum_walmart.txt":             "FILE_ID",
    # add more as needed...
}

print(f"{len(TXT_FILE_IDS)} txt file IDs configured")

Mounted at /content/drive
Excel exists: False
  amazon/  → 0 txt files found
  target/  → 0 txt files found
  walmart/  → 0 txt files found


In [10]:
# Folder IDs (right-click folder in Drive → Share → Copy link)
DRIVE_FOLDERS = {
    "amazon":  "https://drive.google.com/drive/folders/1Oo_s_euUh7rXWInOm0xX8BEw1Wz6MCBD",
    "target":  "https://drive.google.com/drive/folders/1FKE76dcpfJLbMVnwyu9A717UVrV7EHYx",
    "walmart": "https://drive.google.com/drive/folders/1Cv3f-zf-K4DleCWl0sR9zgkbBNRO8Wlq",
}

## 4. Download Files from Google Drive

In [9]:
def drive_download(file_id: str, dest: Path) -> bool:
    if dest.exists():
        log.info(f"  Already downloaded: {dest.name}")
        return True
    url = f"https://drive.google.com/drive/folders/1Cv3f-zf-K4DleCWl0sR9zgkbBNRO8Wlq"
    try:
        gdown.download(url, str(dest), quiet=True)
        log.info(f" {dest.name}")
        return True
    except Exception as e:
        log.warning(f" {dest.name}: {e}")
        return False


# Download Excel
EXCEL_PATH = LOCAL_DIR / "product_list.xlsx"
drive_download(EXCEL_FILE_ID, EXCEL_PATH)

# Download all txt files
for filename, file_id in TXT_FILE_IDS.items():
    drive_download(file_id, LOCAL_DIR / filename)

print(f"\n Downloads complete — files in {LOCAL_DIR}")

/usr/local/lib/python3.12/dist-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=None
  warnings.warn(



 Downloads complete — files in /tmp/kb_data


## 5. Utility Helpers

In [ ]:
def get_domain(url: str) -> str:
    try:
        host = urlparse(url).netloc.lower()
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return ""

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()

def polite_sleep():
    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

def _rating(text: str) -> Optional[float]:
    m = re.search(r"(\d+\.?\d*)", text or "")
    return float(m.group(1)) if m else None

## 6. Fetchers

In [ ]:
HEADERS_BASE = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "DNT": "1",
}


def fetch_requests(url: str, timeout: int = 15) -> Optional[str]:
    headers = {**HEADERS_BASE, "User-Agent": ua.random}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        if resp.status_code == 200:
            return resp.text
        log.warning(f"HTTP {resp.status_code} → {url[:60]}")
    except Exception as e:
        log.warning(f"requests error: {e}")
    return None


async def _fetch_playwright_async(url: str, wait_seconds: int = 6) -> Optional[str]:
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(
                headless=True,
                args=["--disable-blink-features=AutomationControlled",
                      "--no-sandbox", "--disable-dev-shm-usage"],
            )
            ctx = await browser.new_context(
                user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 900},
                locale="en-US",
                extra_http_headers={"Accept-Language": "en-US,en;q=0.9"},
            )
            await ctx.add_init_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
            page = await ctx.new_page()
            await page.route("**/*.{png,jpg,jpeg,gif,svg,woff,woff2}", lambda r: r.abort())
            await page.goto(url, timeout=45000, wait_until="domcontentloaded")
            await page.wait_for_timeout(wait_seconds * 1000)
            html = await page.content()
            await browser.close()
            return html
    except Exception as e:
        log.warning(f"Playwright error: {e}")
    return None

def fetch_playwright(url: str, wait_seconds: int = 6) -> Optional[str]:
    return asyncio.get_event_loop().run_until_complete(_fetch_playwright_async(url, wait_seconds))


def fetch_reddit_api(url: str) -> Optional[dict]:
    try:
        json_url = url.rstrip("/") + ".json?limit=100"
        headers  = {**HEADERS_BASE, "User-Agent": "Mozilla/5.0 (research-scraper/1.0)"}
        resp     = httpx.get(json_url, headers=headers, follow_redirects=True, timeout=15)
        log.info(f"  Reddit API → HTTP {resp.status_code}")
        if resp.status_code != 200:
            return None
        data        = resp.json()
        post        = data[0]["data"]["children"][0]["data"]
        comments    = data[1]["data"]["children"]
        description = clean_text(post.get("title", "") + " " + (post.get("selftext", "") or ""))
        reviews     = []

        def extract_comments(children):
            for child in children:
                d    = child.get("data", {})
                body = clean_text(d.get("body", ""))
                if body and len(body) > 30 and body not in ("[deleted]", "[removed]"):
                    reviews.append({"title": "", "body": body[:1000], "rating": None, "source": "reddit"})
                    if MAX_REVIEWS_PER_SOURCE and len(reviews) >= MAX_REVIEWS_PER_SOURCE:
                        return
                replies = d.get("replies", "")
                if isinstance(replies, dict):
                    extract_comments(replies.get("data", {}).get("children", []))

        extract_comments(comments)
        log.info(f"    Reddit parsed: {len(reviews)} comments, desc {len(description)} chars")
        return {"description": description, "reviews": reviews}
    except Exception as e:
        log.warning(f"Reddit API error: {e}")
        return None


def fetch_amazon_reviews(url: str) -> Optional[dict]:
    """Scrapes /dp/ product page for description + up to 8 inline reviews."""
    m        = re.search(r'/(?:dp|gp/product)/([A-Z0-9]{10})', url)
    asin     = m.group(1) if m else None
    page_url = f"https://www.amazon.com/dp/{asin}?th=1" if asin else url
    html     = fetch_playwright(page_url, wait_seconds=8)
    if not html:
        return None
    soup        = BeautifulSoup(html, "lxml")
    description = ""
    bullets     = soup.select("#feature-bullets li span.a-list-item")
    if bullets:
        description = " ".join(clean_text(b.get_text()) for b in bullets)
    reviews = []
    for r in soup.select("div[data-hook='review']"):
        title_el  = r.select_one("a[data-hook='review-title'] span:not(.a-icon-alt)")
        body_el   = r.select_one("span[data-hook='review-body'] span")
        rating_el = r.select_one("i[data-hook='review-star-rating'] span, i[data-hook='cmps-review-star-rating'] span")
        body   = clean_text(body_el.get_text())  if body_el   else ""
        title  = clean_text(title_el.get_text()) if title_el  else ""
        rating = _rating(clean_text(rating_el.get_text())) if rating_el else None
        if body:
            reviews.append({"title": title, "body": body, "rating": rating, "source": "amazon"})
    log.info(f"    Amazon: {len(reviews)} reviews")
    return {"description": description, "reviews": reviews}


def fetch_target(url: str) -> Optional[dict]:
    """Description only — Target reviews API currently returns 503."""
    html = fetch_playwright(url, wait_seconds=4)
    description = ""
    if html:
        soup = BeautifulSoup(html, "lxml")
        for sel in ["[data-test='item-details-description']",
                    "div[data-test='item-description']",
                    "div[class*='DescriptionContainer']",
                    "meta[name='description']"]:
            el = soup.select_one(sel)
            if el:
                description = clean_text(el.get("content", "") or el.get_text())
                break
    return {"description": description, "reviews": []}


def fetch(url: str):
    domain = get_domain(url)
    if   "reddit.com" in domain: return fetch_reddit_api(url)
    elif "amazon.com" in domain: return fetch_amazon_reviews(url)
    elif "target.com" in domain: return fetch_target(url)
    elif any(d in domain for d in PLAYWRIGHT_DOMAINS): return fetch_playwright(url)
    return fetch_requests(url)

## 7. HTML Parser (generic sites)

In [ ]:
def _parse_ldjson_reviews(soup, source_label: str) -> list:
    reviews = []
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data  = json.loads(script.string or "")
            items = data if isinstance(data, list) else [data]
            for item in items:
                reviews_raw = item.get("review") or item.get("reviews") or []
                if isinstance(reviews_raw, dict): reviews_raw = [reviews_raw]
                for r in reviews_raw[:MAX_REVIEWS_PER_SOURCE]:
                    body  = clean_text(str(r.get("reviewBody", "")))
                    title = clean_text(str(r.get("name", "")))
                    ri    = r.get("reviewRating", {})
                    try:
                        rating = float(ri.get("ratingValue")) if isinstance(ri, dict) and ri.get("ratingValue") else None
                    except (TypeError, ValueError):
                        rating = None
                    if body:
                        reviews.append({"title": title, "body": body, "rating": rating, "source": source_label})
        except Exception:
            pass
    return reviews


def parse_generic(soup, domain):
    result = {"description": "", "reviews": []}
    og   = soup.find("meta", attrs={"property": "og:description"})
    meta = soup.find("meta", attrs={"name": "description"})
    if og and og.get("content"):
        result["description"] = clean_text(og["content"])
    elif meta and meta.get("content"):
        result["description"] = clean_text(meta["content"])
    else:
        paras = [clean_text(p.get_text()) for p in soup.find_all("p") if len(p.get_text(strip=True)) > 40]
        result["description"] = " ".join(paras[:5])
    result["reviews"].extend(_parse_ldjson_reviews(soup, domain))
    if not result["reviews"]:
        for sel in [".review-text", ".reviewText", ".review-body",
                    "[class*='review-content']", "[itemprop='reviewBody']"]:
            for el in soup.select(sel)[:MAX_REVIEWS_PER_SOURCE]:
                text = clean_text(el.get_text())
                if len(text) > 30:
                    result["reviews"].append({"title": "", "body": text, "rating": None, "source": domain})
            if result["reviews"]: break
    return result


def parse(html: str, url: str) -> dict:
    return parse_generic(BeautifulSoup(html, "lxml"), get_domain(url))

## 8. TXT Review File Parsers
Each retailer's exported review text has a different format — separate parsers handle each.

In [ ]:
def parse_amazon_txt(text: str) -> list:
    """Format: Name → X.0 out of 5 stars Title → Reviewed in... → Body"""
    reviews = []
    blocks  = re.split(r'\n(?=\S.*\n\d\.0 out of 5 stars)', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if len(lines) < 3: continue
        person = lines[0]
        rating_m = re.search(r'(\d\.?\d*) out of 5 stars', lines[1])
        rating   = float(rating_m.group(1)) if rating_m else None
        title    = re.sub(r'^\d\.?\d* out of 5 stars\s*', '', lines[1])
        noise    = re.compile(r'^(Reviewed in|Verified Purchase|Helpful|Report|\d+ people|Size:|Color:|Customer image)', re.I)
        body_lines = [l for l in lines[2:] if not noise.match(l) and not re.match(r'^\d\.0 out of 5', l)]
        body = " ".join(body_lines).strip()
        if body:
            reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "amazon_txt"})
    return reviews


def parse_target_txt(text: str) -> list:
    """Format: X out of 5 stars → Date → Title → Body → Name"""
    reviews = []
    blocks  = re.split(r'(?=\d out of 5 stars)', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if not lines: continue
        rating_m = re.search(r'(\d) out of 5', lines[0])
        rating   = float(rating_m.group(1)) if rating_m else None
        noise = re.compile(
            r'^(\d out of 5|originally posted|Report review|Did you|Thumbs|Would|Guest review'
            r'|comfort|size|style|width|\d+\.0 out of \d)', re.I)
        clean = [l for l in lines[1:]
                 if not noise.match(l)
                 and not re.match(r'^\d+ (guests?|people)', l)
                 and not re.match(r'^\w+ \d+, \d{4}', l)
                 and not re.match(r'^\d+ \w+ \d{4}', l)]
        if len(clean) < 2: continue
        person = clean[-1] if len(clean[-1]) < 30 and not clean[-1].endswith('.') else ""
        title  = clean[0]
        body   = " ".join(clean[1:-1] if person else clean[1:]).strip()
        if body:
            reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "target_txt"})
    return reviews


def parse_walmart_txt(text: str) -> list:
    """Format: Date → Name → Item details → X out of 5 stars → Title → Body"""
    reviews = []
    blocks  = re.split(r'(?=(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+,\s+\d{4})', text.strip())
    for block in blocks:
        lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
        if len(lines) < 4: continue
        person   = lines[1] if len(lines) > 1 else ""
        rating_m = re.search(r'(\d) out of 5', block)
        rating   = float(rating_m.group(1)) if rating_m else None
        noise    = re.compile(r'^(Item details|Shoe size|Color:|Verified Purchase|Helpful|Report'
                              r'|Walmart Associate|\(\d+\)|\d+ out of 5)', re.I)
        date_re  = re.compile(r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d+', re.I)
        clean    = [l for l in lines[2:] if not noise.match(l) and not date_re.match(l)
                    and not re.match(r'^Image \d+', l) and not re.match(r'^View more', l)]
        if not clean: continue
        title = clean[0]
        body  = " ".join(clean[1:]).strip() or title
        reviews.append({"person": person, "title": title, "body": body, "rating": rating, "source": "walmart_txt"})
    return reviews


def parse_txt_file(filepath: Path) -> list:
    """Auto-detect source from filename and parse."""
    name = filepath.name.lower()
    text = filepath.read_text(encoding="utf-8", errors="ignore")
    if "amazon" in name:  return parse_amazon_txt(text)
    if "target" in name:  return parse_target_txt(text)
    if "walmart" in name: return parse_walmart_txt(text)
    log.warning(f"Unknown txt source for {name} — skipping")
    return []

## 9. Single-URL Scraper

In [ ]:
def scrape_url(url: str) -> dict:
    if not url or not str(url).startswith("http"):
        return {"description": "", "reviews": [], "review_count": 0, "error": "invalid or missing URL"}
    log.info(f"  Fetching: {url[:80]}")
    raw = fetch(url)
    polite_sleep()
    if raw is None:
        return {"description": "", "reviews": [], "review_count": 0, "error": "fetch failed"}
    result = raw if isinstance(raw, dict) else parse(raw, url)
    result["url"]          = url
    result["review_count"] = len(result.get("reviews", []))
    log.info(f"    desc: {len(result.get('description',''))} chars | reviews: {result['review_count']}")
    return result

## 10. Load Product List from Excel

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name="df_input", dtype=str).fillna("")
df.columns = [c.strip() for c in df.columns]
print(f"Loaded {len(df)} rows | Columns: {list(df.columns)}")

# Group by product: {product_name: {label: url, ...}}
products_grouped = {}
for _, row in df.iterrows():
    name  = row["PRODUCT NAME"].strip()
    label = row["LABEL"].strip()
    link  = row["LINK"].strip()
    products_grouped.setdefault(name, {})[label] = link

# Also capture txt_file_name column, grouped by product
# Strips empty values and deduplicates
txt_files_by_product = {}
for _, row in df.iterrows():
    name     = row["PRODUCT NAME"].strip()
    txt_name = row.get("txt_file_name", "").strip()
    if txt_name:
        txt_files_by_product.setdefault(name, set()).add(txt_name)

print(f"Found {len(products_grouped)} unique products")
print(f"Products with txt files: {sum(1 for v in txt_files_by_product.values() if v)}")
df.head(10)

## 11. Phase 1 — Scrape Descriptions & Reddit
> Hits Amazon (description), Actual Product site, Target (description), and Reddit for each product.
> Set `TEST_MODE = False` to run all products.

In [ ]:
TEST_MODE   = True
SAMPLE_SIZE = 2
DESC_PRIORITY = ["Amazon Link", "Actual Product", "Target", "Reddit / Reviews"]

knowledge_base = {}
to_scrape = list(products_grouped.items())[:SAMPLE_SIZE] if TEST_MODE else list(products_grouped.items())
print(f"Scraping {'TEST: ' + str(SAMPLE_SIZE) if TEST_MODE else 'ALL ' + str(len(to_scrape))} products...\n")

for product_name, links in tqdm(to_scrape, desc="Products"):
    print(f"\n{'='*60}\n▶ {product_name}\n{'='*60}")
    kb_entry = {
        "product_name":         product_name,
        "sources":              {},
        "combined_description": "",
        "all_reviews":          [],
    }

    for label, url in links.items():
        if not url or url == "—": continue
        print(f"  [{label}]")
        scraped = scrape_url(url)
        kb_entry["sources"][label] = {
            "url":          url,
            "label":        label,
            "description":  scraped.get("description", ""),
            "reviews":      scraped.get("reviews", []),
            "review_count": scraped.get("review_count", 0),
            "error":        scraped.get("error"),
        }
        kb_entry["all_reviews"].extend(scraped.get("reviews", []))

    # Pick best available description
    for lbl in DESC_PRIORITY:
        if kb_entry["sources"].get(lbl, {}).get("description"):
            kb_entry["combined_description"] = kb_entry["sources"][lbl]["description"]
            break
    if not kb_entry["combined_description"]:
        kb_entry["combined_description"] = " | ".join(
            s["description"] for s in kb_entry["sources"].values() if s.get("description")
        )

    kb_entry["total_reviews"] = len(kb_entry["all_reviews"])
    knowledge_base[product_name] = kb_entry
    print(f"  ✓ Web reviews collected: {kb_entry['total_reviews']}")

print(f"\n Phase 1 done — {len(knowledge_base)} products scraped")

## 12. Phase 2 — Inject TXT Reviews from Drive
Reads each product's txt files (listed in the `txt_file_name` column of the Excel),
parses them, and appends the reviews into the matching knowledge base entry.

In [ ]:
txt_summary = []

for product_name, txt_filenames in txt_files_by_product.items():
    # Only inject for products that were scraped (respects TEST_MODE)
    if product_name not in knowledge_base:
        continue

    entry = knowledge_base[product_name]
    print(f"\n▶ {product_name}")

    for filename in sorted(txt_filenames):
        filepath = LOCAL_DIR / filename
        if not filepath.exists():
            log.warning(f" Not found: {filename} (was it downloaded in step 4?)")
            continue

        reviews = parse_txt_file(filepath)
        source_key = filename.replace(".txt", "")  # e.g. crocs_amazon

        entry["sources"][source_key] = {
            "url":          str(filepath),
            "label":        source_key,
            "description":  "",
            "reviews":      reviews,
            "review_count": len(reviews),
            "error":        None,
        }
        entry["all_reviews"].extend(reviews)
        print(f"  [{source_key}] → {len(reviews)} reviews loaded")
        txt_summary.append({"product": product_name, "file": filename, "reviews": len(reviews)})

    entry["total_reviews"] = len(entry["all_reviews"])
    print(f"  ✓ Total reviews now: {entry['total_reviews']}")

print(f"\n Phase 2 done")

## 13. Save Knowledge Base to JSON

In [ ]:
out_path = Path(OUTPUT_JSON)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, indent=2, ensure_ascii=False)

total_reviews = sum(e['total_reviews'] for e in knowledge_base.values())
print(f" Saved → {out_path}")
print(f"   Products     : {len(knowledge_base)}")
print(f"   Total reviews: {total_reviews}")

## 14. Inspect Results

In [ ]:
rows = []
for name, entry in knowledge_base.items():
    for label, src in entry["sources"].items():
        rows.append({
            "Product":         name,
            "Source":          label,
            "Reviews Found":   src["review_count"],
            "Has Description": bool(src["description"]),
            "Error":           src.get("error") or "",
        })
pd.DataFrame(rows)

In [ ]:
# Deep-dive into one product
entry = knowledge_base[list(knowledge_base.keys())[0]]
print(f"Product      : {entry['product_name']}")
print(f"Total reviews: {entry['total_reviews']}")
print(f"\nDescription:\n{entry['combined_description'][:400]}")
print(f"\nFirst 5 reviews:")
for r in entry["all_reviews"][:5]:
    person = r.get('person', '')
    print(f"  [{r['source']}] {r['rating']} {person} | {r['title']} — {r['body'][:100]}")

## Debug *(optional)*

In [ ]:
DEBUG_URL = "https://www.reddit.com/r/scacjdiscussion/comments/izbqbn/hot_take_cerave_moisturizing_cream_in_the_tub_is/"

raw = fetch(DEBUG_URL)
if isinstance(raw, dict):
    print(f"Pre-parsed dict — {len(raw['reviews'])} reviews")
    print(f"Description: {raw['description'][:200]}")
    for r in raw['reviews'][:3]:
        print(f"  - {r['body'][:120]}")
else:
    soup = BeautifulSoup(raw, "lxml")
    with open("debug_output.html", "w") as f: f.write(raw)
    print(f"HTML — {len(raw)} chars")
    print(soup.get_text()[:2000])